Standarize Dates and Data types

In [0]:
from pyspark.sql.functions import to_timestamp, col

orders_df = spark.table("uc_quickbite.bronze_ingestion.fact_orders") \
    .withColumn(
        "order_timestamp",
        to_timestamp("order_timestamp")
    ) \
    .withColumn(
        "is_cancelled",
        col("is_cancelled").cast("string")
    )


Filter Invalid / Noisy Records
[Invalid and zero-value transactions were excluded to avoid skewing revenue KPIs.]

In [0]:
orders_clean_df = orders_df.filter(
  (col("total_amount")>0) &
  (col("order_id").isNotNull())
)

Create Crisis Phase Flag

In [0]:
from pyspark.sql.functions import col, when, to_timestamp

orders_df = spark.table("uc_quickbite.bronze_ingestion.fact_orders") \
    .select(
        "order_id",
        "customer_id",
        "restaurant_id",
        "delivery_partner_id",
        to_timestamp("order_timestamp").alias("order_timestamp"),
        "subtotal_amount",
        "discount_amount",
        "delivery_fee",
        "total_amount",
        "is_cancelled"
    )

orders_phase_df = orders_df.withColumn(
    "crisis_phase",
    when(col("order_timestamp") < "2025-06-01", "Pre-Crisis")
    .when(col("order_timestamp").between("2025-06-01", "2025-09-30"), "Crisis")
    .otherwise("Recovery")
)


Joining Customer & Restaurant Dimensions [To Enable city-level and cuisine-level insights]

In [0]:
customer_df = spark.table("uc_quickbite.bronze_ingestion.dim_customer")
restaurant_df = spark.table("uc_quickbite.bronze_ingestion.dim_restaurant")

orders_enriched_df = orders_phase_df \
    .join(customer_df, "customer_id", "left") \
    .join(restaurant_df, "restaurant_id", "left")


Delivery SLA & Delay Metrics [To Direct driver of churn & sentiment]

In [0]:
delivery_df = spark.table("uc_quickbite.bronze_ingestion.fact_deliveryPerformance")

orders_delivery_df = orders_enriched_df.join(
    delivery_df, "order_id", "left"
).withColumn(
    "delivery_delay_mins",
    col("actual_delivery_time_mins") - col("expected_delivery_time_mins")
).withColumn(
    "sla_breached",
    when(col("delivery_delay_mins") > 0, 1).otherwise(0)
)


Revenue Derivations [Accurate revenue impact analysis]

In [0]:
orders_revenue_df = orders_delivery_df.withColumn(
    "net_revenue",
    col("subtotal_amount") - col("discount_amount") + col("delivery_fee")
)

Cancellation Reason Flags [To Link cancellations to delivery failures]

In [0]:
orders_cancel_flag_df = orders_revenue_df.withColumn(
    "delivery_related_cancel",
    when(
        (col("is_cancelled") == "Y") &
        (col("delivery_delay_mins") > 10),
        1
    ).otherwise(0)
)

Ratings + Sentiment Enrichment [To enable sentiment driven analysis]

In [0]:
ratings_df = spark.table("uc_quickbite.bronze_ingestion.fact_ratings")

ratings_enriched_df = ratings_df.withColumn(
    "sentiment_bucket",
    when(col("sentiment_score") > 0.2, "Positive")
    .when(col("sentiment_score") < -0.2, "Negative")
    .otherwise("Neutral")
).withColumn(
    "rating_bucket",
    when(col("rating") >= 4, "High")
    .when(col("rating") >= 3, "Medium")
    .otherwise("Low")
)

Customer Behavior Metrics

In [0]:
from pyspark.sql.functions import count, avg

customer_behavior_df = orders_cancel_flag_df.groupBy("customer_id").agg(
    count("order_id").alias("total_orders"),
    avg("net_revenue").alias("avg_order_value"),
    avg("delivery_delay_mins").alias("avg_delay"),
    avg("sla_breached").alias("sla_breach_rate")
)

write transformations to silver table

In [0]:
orders_cancel_flag_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("uc_quickbite.silver_transform.orders_enriched")

ratings_enriched_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("uc_quickbite.silver_transform.ratings_enriched")

customer_behavior_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("uc_quickbite.silver_transform.customer_behavior")
